# Tam Kan Sayımı (CBC) ile Otoimmün Bozukluk Erken Teşhis Modeli

Bu notebook çalışması, projenin genel kapsamını ve teşhis yeteneklerini bir üst seviyeye taşımak amacıyla kurgulanmıştır. Projenin ilk aşamalarında geliştirilen spesifik `anemia_model` ve `diabetes_model` yapılarından bağımsız olarak bu çalışma; tam kan sayımı (CBC) laboratuvar parametrelerini kullanarak kişilerin otoimmün hastalık risklerini ya da sağlıklı olma durumlarını tahmin etmeyi amaçlar.

### Bu Model Backend Mimarisine Nasıl Entegre Olacak?
Günün sonunda elde edilecek `autoimmune_model.pkl` dosyası, backend tarafında üçüncü bir büyük bağımsız modül olarak konumlandırılacaktır. Kullanıcı sisteme standart bir tam kan tahlili raporu girdiğinde, sistem arka planda hem anemi modelini hem de bu çoklu sınıflandırma yapan otoimmün modelini aynı anda tetikleyerek tek bir tahlilden maksimum klinik çıkarım üretecektir.

Dataset: https://www.kaggle.com/datasets/abdullahragheb/all-autoimmune-disorder-10k

In [69]:
!pip install pandas numpy seaborn matplotlib scikit-learn xgboost lightgbm catboost

Defaulting to user installation because normal site-packages is not writeable
  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl.metadata (17 kB)
  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.5/69.5 MB 741.5 kB/s eta 0:01:34
   ---------------------------------------- 0.5/69.5 MB 741.5 kB/s eta 0:01:34
   ---------------------------------------- 0.8/69.5 MB 708.0 kB/s eta 0:01:38
   ---------------------------------------- 0.8/69.5 MB 708.0 kB/s eta 0:01:38
    --------------------------------------- 1.0/69.5 MB 622.0 kB/s eta 0:01:51
    --------------------------------------- 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pickle

In [2]:
df=pd.read_csv("Final_Balanced_Autoimmune_Disorder_Dataset.csv")
pd.set_option('display.max_columns', None)
print("Veri Setindeki Tüm Sütunlar:")
print(df.columns.tolist())
print("\nVeri Tipleri Özeti:")
print(df.dtypes.value_counts())
print("\n")
display(df.head())
print("\n")
display(df.describe())
print("\n")
display(df["Diagnosis"].unique())

Veri Setindeki Tüm Sütunlar:
['Patient_ID', 'Age', 'Gender', 'Diagnosis', 'Sickness_Duration_Months', 'RBC_Count', 'Hemoglobin', 'Hematocrit', 'MCV', 'MCH', 'MCHC', 'RDW', 'Reticulocyte_Count', 'WBC_Count', 'Neutrophils', 'Lymphocytes', 'Monocytes', 'Eosinophils', 'Basophils', 'PLT_Count', 'MPV', 'ANA', 'Esbach', 'MBL_Level', 'ESR', 'C3', 'C4', 'CRP', 'Anti-dsDNA', 'Anti-Sm', 'Rheumatoid factor', 'ACPA', 'Anti-TPO', 'Anti-Tg', 'Anti-SMA', 'Low-grade fever', 'Fatigue or chronic tiredness', 'Dizziness', 'Weight loss', 'Rashes and skin lesions', 'Stiffness in the joints', 'Brittle hair or hair loss', 'Dry eyes and/or mouth', "General 'unwell' feeling", 'Joint pain', 'Anti_dsDNA', 'Anti_enterocyte_antibodies', 'anti_LKM1', 'Anti_RNP', 'ASCA', 'Anti_Ro_SSA', 'Anti_CBir1', 'Anti_BP230', 'Anti_tTG', 'DGP', 'Anti_BP180', 'ASMA', 'Anti_IF', 'IgG_IgE_receptor', 'Anti_SRP', 'Anti_desmoglein_3', 'Anti_La_SSB', 'Anti_Jo1', 'ANCA', 'anti_centromere', 'Anti_desmoglein_1', 'EMA', 'Anti_type_VII_collag

,Patient_ID,Age,Gender,Diagnosis,Sickness_Duration_Months,RBC_Count,Hemoglobin,Hematocrit,MCV,MCH,MCHC,RDW,Reticulocyte_Count,WBC_Count,Neutrophils,Lymphocytes,Monocytes,Eosinophils,Basophils,PLT_Count,MPV,ANA,Esbach,MBL_Level,ESR,C3,C4,CRP,Anti-dsDNA,Anti-Sm,Rheumatoid factor,ACPA,Anti-TPO,Anti-Tg,Anti-SMA,Low-grade fever,Fatigue or chronic tiredness,Dizziness,Weight loss,Rashes and skin lesions,Stiffness in the joints,Brittle hair or hair loss,Dry eyes and/or mouth,General 'unwell' feeling,Joint pain,Anti_dsDNA,Anti_enterocyte_antibodies,anti_LKM1,Anti_RNP,ASCA,Anti_Ro_SSA,Anti_CBir1,Anti_BP230,Anti_tTG,DGP,Anti_BP180,ASMA,Anti_IF,IgG_IgE_receptor,Anti_SRP,Anti_desmoglein_3,Anti_La_SSB,Anti_Jo1,ANCA,anti_centromere,Anti_desmoglein_1,EMA,Anti_type_VII_collagen,C1_inhibitor,Anti_TIF1,Anti_epidermal_basement_membrane_IgA,Anti_OmpC,pANCA,Anti_tissue_transglutaminase,anti_Scl_70,Anti_Mi2,Anti_parietal_cell,Progesterone_antibodies,Anti_Sm
0,6,62,Male,Autoimmune orchitis,41,4.75,13.37,43.11,101.91,28.41,34.98,12.12,2.77,4458,40.05,34.54,3.29,4.42,0.83,402793,8.45,1,2.93,0.71,22,0.93,0.12,6.72,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0,1,0,0,1,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,20,54,Female,Autoimmune orchitis,41,4.32,10.76,39.92,95.96,28.22,31.70,12.89,2.98,4974,32.30,17.21,2.11,4.24,1.44,145389,7.22,0,2.47,1.58,41,0.94,0.56,2.67,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1,0,1,0,1,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,46,34,Male,Autoimmune orchitis,86,4.42,11.91,38.38,80.56,28.40,35.21,12.73,1.35,8766,32.30,20.94,5.81,4.53,1.33,111764,11.63,0,1.18,1.18,42,0.88,0.27,8.14,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1,0,0,1,1,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,108,22,Male,Autoimmune orchitis,43,4.33,12.72,39.99,84.71,26.67,31.25,14.62,1.63,8828,34.62,25.83,3.30,2.30,0.81,276336,8.40,1,2.46,0.95,29,1.04,0.13,2.87,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0,1,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,142,20,Female,Autoimmune orchitis,50,3.99,11.07,43.58,89.87,30.64,32.77,14.45,2.12,4583,56.56,42.86,7.51,2.30,1.24,297272,9.73,0,2.68,1.37,47,1.30,0.12,2.26,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0,1,1,1,1,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


,Patient_ID,Age,Sickness_Duration_Months,RBC_Count,Hemoglobin,Hematocrit,MCV,MCH,MCHC,RDW,Reticulocyte_Count,WBC_Count,Neutrophils,Lymphocytes,Monocytes,Eosinophils,Basophils,PLT_Count,MPV,ANA,Esbach,MBL_Level,ESR,C3,C4,CRP,Anti-dsDNA,Anti-Sm,Rheumatoid factor,ACPA,Anti-TPO,Anti-Tg,Anti-SMA,Low-grade fever,Fatigue or chronic tiredness,Dizziness,Weight loss,Rashes and skin lesions,Stiffness in the joints,Brittle hair or hair loss,Dry eyes and/or mouth,General 'unwell' feeling,Joint pain,Anti_dsDNA,Anti_enterocyte_antibodies,anti_LKM1,Anti_RNP,ASCA,Anti_Ro_SSA,Anti_CBir1,Anti_BP230,Anti_tTG,DGP,Anti_BP180,ASMA,Anti_IF,IgG_IgE_receptor,Anti_SRP,Anti_desmoglein_3,Anti_La_SSB,Anti_Jo1,ANCA,anti_centromere,Anti_desmoglein_1,EMA,Anti_type_VII_collagen,C1_inhibitor,Anti_TIF1,Anti_epidermal_basement_membrane_IgA,Anti_OmpC,pANCA,Anti_tissue_transglutaminase,anti_Scl_70,Anti_Mi2,Anti_parietal_cell,Progesterone_antibodies,Anti_Sm
count,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.0,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.000000,11712.0,11712.000000
mean,4980.190403,49.332223,62.094348,4.273274,12.508963,40.629464,89.718165,29.148669,33.386141,14.042355,1.889914,8006.484887,52.633982,29.736572,5.990416,2.992428,0.983379,293209.682462,9.451584,0.714566,1.779883,1.272425,28.911714,0.958088,0.343085,5.376245,0.508880,0.532787,0.539361,0.544399,0.433572,0.449710,0.501622,0.530994,0.487449,0.502732,0.530225,0.505123,0.512124,0.507258,0.490864,0.522370,0.494536,0.003501,0.001708,0.001964,0.001110,0.004952,0.001110,0.004098,0.008453,0.0,0.001281,0.008453,0.001964,0.001452,0.000854,0.002818,0.000598,0.001110,0.002818,0.004098,0.002988,0.000598,0.001281,0.003159,0.004525,0.002818,0.001366,0.004098,0.000854,0.005891,0.002988,0.002818,0.001452,0.0,0.003501
std,2965.242746,17.307711,33.779429,0.444560,1.464328,2.553632,8.445325,1.707619,1.434252,1.181453,0.622830,2360.585827,12.833394,8.745286,2.272708,1.131568,0.281841,117730.603833,1.353694,0.451640,0.737794,0.432436,11.887982,0.305104,0.145977,2.577840,0.499942,0.498945,0.498470,0.498046,0.495589,0.497486,0.500019,0.499060,0.499864,0.500014,0.499107,0.499995,0.499874,0.499969,0.499938,0.499521,0.499991,0.059065,0.041290,0.044273,0.033299,0.070200,0.033299,0.063890,0.091554,0.0,0.035766,0.091554,0.044273,0.038073,0.029209,0.053009,0.024441,0.033299,0.053009,0.063890,0.054587,0.024441,0.035766,0.056120,0.067121,0.053009,0.036937,0.063890,0.029209,0.076532,0.054587,0.053009,0.038073,0.0,0.059065
min,6.000000,20.000000,1.000000,3.510000,10.000000,36.000000,75.090000,26.000000,31.000000,12.000000,0.800000,4004.000000,30.070000,15.260000,2.020000,1.010000,0.500000,100073.000000,7.010000,0.000000,0.500000,0.500000,1.000000,0.400000,0.100000,0.200000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00

array(['Autoimmune orchitis', 'Rheumatoid arthritis', 'Sjögren syndrome',
       "Graves' disease", 'Systemic lupus erythematosus (SLE)', 'Other'],
      dtype=object)

## Özellik Seçimi (Feature Selection) ve Veri Ön İşleme

Veri setinin genel yapısı incelendiğinde, laboratuvar ortamında yapılan çok spesifik ve ileri düzey antikor testleri ile fiziksel semptomları içeren 79 farklı sütun olduğu görülmüştür. Ancak projenin temel vizyonu, kullanıcının evde eline aldığı **standart bir rutin tam kan sayımı (CBC) ve rutin biyokimya** raporuyla sistemden sonuç alabilmesidir. 

Bu doğrultuda, rutin laboratuvar panellerinde yer almayan ileri düzey uzmanlık antikorları elenmiş ve modelin girdileri (features) standart bir kan panelini simüle edecek şekilde aşağıdaki alt gruplara daraltılmıştır:

* **Temel Demografi:** `Age`, `Gender`
* **Kırmızı Kan Hücreleri (Eritrosit Paneli - Anemi & Oksijen Taşınımı):** `RBC_Count`, `Hemoglobin`, `Hematocrit`, `MCV`, `MCH`, `MCHC`, `RDW`, `Reticulocyte_Count`
* **Beyaz Kan Hücreleri (Lökosit Paneli - Bağışıklık & Enfeksiyon):** `WBC_Count`, `Neutrophils`, `Lymphocytes`, `Monocytes`, `Eosinophils`, `Basophils`
* **Trombosit Paneli (Pıhtılaşma Sistemi):** `PLT_Count`, `MPV`
* **Rutin İnflamasyon Paneli (Sistemik Yanıt):** `CRP`, `ESR`
* **Hedef Değişken (Sınıflandırma):** `Diagnosis`

Bu filtreleme işlemi, modelin backend tarafında standart girdilerle çalışmasını sağlayarak üretim ortamındaki (production) uygulanabilirliğini maksimuma çıkaracaktır.

In [3]:
selected_features = [
    "Age", "Gender",
    "RBC_Count", "Hemoglobin", "Hematocrit", "MCV", "MCH", "MCHC", "RDW", "Reticulocyte_Count",
    "WBC_Count", "Neutrophils", "Lymphocytes", "Monocytes", "Eosinophils", "Basophils",
    "PLT_Count", "MPV",
    "CRP", "ESR",
    "Diagnosis"
]
df_model = df[selected_features].copy()
display(df_model)

,Age,Gender,RBC_Count,Hemoglobin,Hematocrit,MCV,MCH,MCHC,RDW,Reticulocyte_Count,WBC_Count,Neutrophils,Lymphocytes,Monocytes,Eosinophils,Basophils,PLT_Count,MPV,CRP,ESR,Diagnosis
0,62,Male,4.75,13.37,43.11,101.91,28.41,34.98,12.12,2.77,4458,40.05,34.54,3.29,4.42,0.83,402793,8.45,6.72,22,Autoimmune orchitis
1,54,Female,4.32,10.76,39.92,95.96,28.22,31.70,12.89,2.98,4974,32.30,17.21,2.11,4.24,1.44,145389,7.22,2.67,41,Autoimmune orchitis
2,34,Male,4.42,11.91,38.38,80.56,28.40,35.21,12.73,1.35,8766,32.30,20.94,5.81,4.53,1.33,111764,11.63,8.14,42,Autoimmune orchitis
3,22,Male,4.33,12.72,39.99,84.71,26.67,31.25,14.62,1.63,8828,34.62,25.83,3.30,2.30,0.81,276336,8.40,2.87,29,Autoimmune orchitis
4,20,Female,3.99,11.07,43.58,89.87,30.64,32.77,14.45,2.12,4583,56.56,42.86,7.51,2.30,1.24,297272,9.73,2.26,47,Autoimmune orchitis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11707,40,Female,4.57,13.77,40.85,93.25,27.12,32.12,14.15,3.00,9395,68.01,21.88,3.53,3.55,0.79,324445,11.58,7.75,46,Other
11708,20,Female,4.14,12.35,42.15,86.16,29.15,32.56,15.15,1.31,5779,64.51,15.70,3.22,3.39,0.84,126130,11.38,5.76,26,Other
11709,27,Male,3.85,11.26,38.35,79.00,26.59,34.70,13.34,0.99,7505,69.10,27.18,4.98,2.21,0.81,105735,10.26,3.45,21,Other
11710,76,Male,4.85,12.32,37.83,78.01,29.99,32.63,12.01,2.17,8909,72.88,36.86,6.79,2.06,1.24,454428,8.06,3.34,17,Other


In [23]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from catboost import CatBoostClassifier
import xgboost as xgb
import lightgbm as lgb
import time

In [5]:
df_model["Gender"]=df_model["Gender"].map({"Male" : 0, "Female" : 1})
encoder=LabelEncoder()
df_model["Diagnosis"]=encoder.fit_transform(df_model["Diagnosis"])
class_mapping = dict(zip(encoder.transform(encoder.classes_), encoder.classes_)) #Backend'de kullanmak için eşleşmeleri loglama
print("Sınıf Eşleşmeleri (Target Mapping):")
for num, disease in class_mapping.items():
    print(f"  {num} -> {disease}")
print("-" * 50)
display(df_model.head())
print(df_model.dtypes.value_counts())

Sınıf Eşleşmeleri (Target Mapping):
  0 -> Autoimmune orchitis
  1 -> Graves' disease
  2 -> Other
  3 -> Rheumatoid arthritis
  4 -> Sjögren syndrome
  5 -> Systemic lupus erythematosus (SLE)
--------------------------------------------------


,Age,Gender,RBC_Count,Hemoglobin,Hematocrit,MCV,MCH,MCHC,RDW,Reticulocyte_Count,WBC_Count,Neutrophils,Lymphocytes,Monocytes,Eosinophils,Basophils,PLT_Count,MPV,CRP,ESR,Diagnosis
0,62,0,4.75,13.37,43.11,101.91,28.41,34.98,12.12,2.77,4458,40.05,34.54,3.29,4.42,0.83,402793,8.45,6.72,22,0
1,54,1,4.32,10.76,39.92,95.96,28.22,31.70,12.89,2.98,4974,32.30,17.21,2.11,4.24,1.44,145389,7.22,2.67,41,0
2,34,0,4.42,11.91,38.38,80.56,28.40,35.21,12.73,1.35,8766,32.30,20.94,5.81,4.53,1.33,111764,11.63,8.14,42,0
3,22,0,4.33,12.72,39.99,84.71,26.67,31.25,14.62,1.63,8828,34.62,25.83,3.30,2.30,0.81,276336,8.40,2.87,29,0
4,20,1,3.99,11.07,43.58,89.87,30.64,32.77,14.45,2.12,4583,56.56,42.86,7.51,2.30,1.24,297272,9.73,2.26,47,0


float64    15
int64       5
int32       1
Name: count, dtype: int64


In [18]:
X=df_model.drop(columns=["Diagnosis"])
y=df_model["Diagnosis"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [27]:
models={
    "Random Forest Classifier":RandomForestClassifier(random_state=42),
    "Support Vector Classifier":make_pipeline(StandardScaler(),
                                              SVC(random_state=42)),
    "K-Neighbors Classifier":make_pipeline(StandardScaler(),
                                           KNeighborsClassifier()),
    "CatBoost":CatBoostClassifier(random_state=42, verbose=0),
    "XGBoost":xgb.XGBClassifier(random_state=42, verbosity=0),
    "LightGBM":lgb.LGBMClassifier(random_state=42, verbose=-1)
}

In [30]:
result_list=[]

for name, model in models.items():
    st_time=time.time()
    model.fit(X_train, y_train)
    e_time=time.time()
    y_pred = model.predict(X_test)
    
    train_acc = model.score(X_train, y_train)
    test_acc = model.score(X_test, y_test)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
    
    cm = confusion_matrix(y_test, y_pred)
    cr = classification_report(y_test, y_pred)

    result_list.append({
        "Model Name": name,
        "Train Accuracy": round(train_acc, 4),
        "Test Accuracy": round(test_acc, 4),
        "5-Fold CV Mean": round(cv_scores.mean(), 4),
        "Training Length(in seconds)": round(e_time-st_time, 3)
    })

C:\Users\Bariscan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Bariscan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Bariscan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war

In [31]:
df_results = pd.DataFrame(result_list)
df_results = df_results.sort_values(by="Test Accuracy", ascending=False).reset_index(drop=True)
display(df_results)

,Model Name,Train Accuracy,Test Accuracy,5-Fold CV Mean,Training Length(in seconds)
0,Random Forest Classifier,0.9790,0.9782,0.9763,0.846
1,CatBoost,0.9790,0.9782,0.9757,9.161
2,LightGBM,0.9790,0.9782,0.9748,0.436
3,Support Vector Classifier,0.9778,0.9774,0.9738,0.965
4,XGBoost,0.9790,0.9774,0.9758,0.556
5,K-Neighbors Classifier,0.9761,0.9761,0.9640,0.005
